# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import duckdb
import os
from getpass import getpass

# Enter your Hugging Face READ token privately
HF_TOKEN = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

df

Enter your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,78835655,2025-01-27,2026-06-30


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

One row represents one client × content item × report date observation.

The available time window is verified by the query below using the minimum and maximum `report_date`.

The proposed grain is verified by checking for duplicate `client_id × content_id × report_date` combinations.

In [24]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
SELECT *
FROM read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
LIMIT 5
""").df()

display(test)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [26]:
# ML-04 — Section 1: Unit of analysis + time window

warehouse = "hf://datasets/FlyRank/internship-warehouse"

# Read the fact table once for this section
fact = f"""
read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

# ------------------------------------------------------------
# 1. Table size and date window
# ------------------------------------------------------------
section1_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT report_date) AS dates,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {fact}
""").df()

print("TABLE SUMMARY")
display(section1_summary)


# ------------------------------------------------------------
# 2. Verify grain
# ------------------------------------------------------------
# Expected grain:
# client_hash_id × content_hash_id × report_date

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {fact}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("\nGRAIN CHECK")
print("Duplicate combinations found:", len(grain_check))
display(grain_check)

if len(grain_check) == 0:
    print(
        "PASS: One row = one client × content item × report date."
    )
else:
    print(
        "WARNING: Duplicate client × content × report date combinations found."
    )

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TABLE SUMMARY


,total_rows,clients,content_items,dates,first_date,last_date
0,78835655,70,427292,520,2025-01-27,2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


GRAIN CHECK
Duplicate combinations found: 5


,client_hash_id,content_hash_id,report_date,row_count
0,client_b77d0d5f08f05e64,content_1b18ea04806f1278,2026-06-13,2
1,client_8ddc46da5414ffd8,content_a9789f58505f4f9b,2026-06-13,2
2,client_8ddc46da5414ffd8,content_d07f96c572dc6f2a,2026-06-13,2
3,client_b77d0d5f08f05e64,content_6b6a532cb09428b2,2026-06-20,2
4,client_b77d0d5f08f05e64,content_52a40d3e61e4e4cc,2026-06-20,2


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature
Historical search-performance and content-performance fields that are available before the prediction/outcome window and are not used to construct the label.

### Label / proxy
The future outcome used to evaluate the content-refresh prioritization task. Any field used to calculate the outcome is treated as a label/proxy and is not used as a feature.

### Context
- `client_hash_id`
- `content_hash_id`
- `report_date`

These fields are used for grouping, joining, filtering, splitting, and interpreting results.

### Excluded
- Future-window measurements — they contain information unavailable at prediction time.
- `trend_direction` — excluded if it directly determines the label.
- `trend_pct` — excluded if it is used to construct the label.
- `client_hash_id` — pseudonymous identifier, not a predictive feature.
- `content_hash_id` — pseudonymous identifier, not a predictive feature.

In [28]:
# ML-04 — Section 2: Fields — feature / label / context / excluded

warehouse = "hf://datasets/FlyRank/internship-warehouse"

# Get the schema directly from the Parquet warehouse
columns_df = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{warehouse}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
""").df()

print("=" * 70)
print("ALL AVAILABLE FIELDS")
print("=" * 70)
display(columns_df)


# ------------------------------------------------------------
# Identify GSC-related fields
# ------------------------------------------------------------
column_names = columns_df["column_name"].tolist()

gsc_columns = [
    c for c in column_names
    if "gsc" in c.lower()
]

print("\n" + "=" * 70)
print("GSC FIELDS")
print("=" * 70)

for col in gsc_columns:
    print("-", col)


# ------------------------------------------------------------
# Identify GA4-related fields
# ------------------------------------------------------------
ga4_columns = [
    c for c in column_names
    if "ga4" in c.lower()
]

print("\n" + "=" * 70)
print("GA4 FIELDS")
print("=" * 70)

for col in ga4_columns:
    print("-", col)


# ------------------------------------------------------------
# Context fields — using the ACTUAL warehouse names
# ------------------------------------------------------------
context_candidates = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

context_fields = [
    c for c in context_candidates
    if c in column_names
]

print("\n" + "=" * 70)
print("CONTEXT FIELDS")
print("=" * 70)

for col in context_fields:
    print("-", col)


# ------------------------------------------------------------
# Potential label / leakage fields
# ------------------------------------------------------------
label_candidates = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

label_fields = [
    c for c in label_candidates
    if c in column_names
]

print("\n" + "=" * 70)
print("LABEL / POTENTIAL LEAKAGE FIELDS")
print("=" * 70)

for col in label_fields:
    print("-", col)


# ------------------------------------------------------------
# Final field classification
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("ML-04 FIELD CLASSIFICATION")
print("=" * 70)

print("""
CONTEXT
-------
client_hash_id
content_hash_id
report_date

LABEL / PROXY
-------------
Fields used to define the future outcome/target.

EXCLUDED
--------
- Future-window fields: prevent target leakage.
- trend_direction: excluded if it defines the label.
- trend_pct: excluded if it is used to calculate the label.
- client_hash_id: pseudonymous identifier; grouping/splitting only.
- content_hash_id: pseudonymous identifier; grouping/joining only.

FEATURES
--------
Historical search/content-performance fields that are known BEFORE
the prediction/outcome window and are not used to construct the label.
""")

ALL AVAILABLE FIELDS


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



GSC FIELDS
- client_has_gsc
- gsc_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position

GA4 FIELDS
- client_has_ga4
- ga4_data_available
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec

CONTEXT FIELDS
- client_hash_id
- content_hash_id
- report_date

LABEL / POTENTIAL LEAKAGE FIELDS

ML-04 FIELD CLASSIFICATION

CONTEXT
-------
client_hash_id
content_hash_id
report_date

LABEL / PROXY
-------------
Fields used to define the future outcome/target.

EXCLUDED
--------
- Future-window fields: prevent target leakage.
- trend_direction: excluded if it defines the label.
- trend_pct: excluded if it is used to calculate the label.
- client_hash_id: pseudonymous identifier; grouping/splitting only.
- content_hash_id: pseudonymous identifier; grouping/joining only.

FEATURES
--------
Historical search/content-performance fields that are known BEFORE
the prediction/outcome window and are not used to construct the label.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The queries verify the proposed grain, total counts, date window, client-level history, access flags, missing values, client-specific missingness, GSC/GA4 history, and the final-month boundary.

The grain claim is supported if the duplicate check returns zero rows.

Missingness is checked overall and by client because missing values may follow systematic patterns.

June 2026 is treated as the final sealed month. March 2026 is used as a mid-panel development month rather than the final-month sample.

In [31]:
# ML-04 — Section 3: Verify claims with queries

warehouse = "hf://datasets/FlyRank/internship-warehouse"

fact = f"""
read_parquet(
    '{warehouse}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

# Get actual columns
columns_df = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {fact}
""").df()

columns = columns_df["column_name"].tolist()


# ============================================================
# A. COUNTS + DATE WINDOW
# ============================================================

print("=" * 70)
print("A. COUNTS AND DATE WINDOW")
print("=" * 70)

summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT report_date) AS dates,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {fact}
""").df()

display(summary)


# ============================================================
# B. GRAIN CHECK
# ============================================================

print("=" * 70)
print("B. GRAIN CHECK")
print("=" * 70)

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {fact}
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate combinations found:", len(grain_check))
display(grain_check)

if len(grain_check) == 0:
    print("PASS: One row = one client × content × report date.")
else:
    print("WARNING: Duplicate combinations found.")


# ============================================================
# C. CLIENT HISTORY
# ============================================================

print("=" * 70)
print("C. CLIENT HISTORY")
print("=" * 70)

client_history = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(*) AS rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {fact}
    GROUP BY client_hash_id
    ORDER BY rows DESC
    LIMIT 20
""").df()

display(client_history)


# ============================================================
# D. GSC / GA4 ACCESS FLAGS
# ============================================================

print("=" * 70)
print("D. GSC / GA4 ACCESS FLAGS")
print("=" * 70)

if "client_has_gsc" in columns and "client_has_ga4" in columns:

    access_check = con.sql(f"""
        SELECT
            client_has_gsc,
            client_has_ga4,
            COUNT(*) AS rows
        FROM {fact}
        GROUP BY
            client_has_gsc,
            client_has_ga4
        ORDER BY
            client_has_gsc,
            client_has_ga4
    """).df()

    display(access_check)
else:
    print("Access flag columns not found.")


# ============================================================
# E. MISSING VALUES
# ============================================================

print("=" * 70)
print("E. MISSING VALUES")
print("=" * 70)

missing_candidates = [
    "gsc_avg_position",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_ctr",
    "sessions",
    "engagement_rate",
    "scroll_rate"
]

available_missing = [
    c for c in missing_candidates
    if c in columns
]

if available_missing:

    expressions = []

    for col in available_missing:
        expressions.append(f"""
            ROUND(
                100.0 * AVG(
                    CASE
                        WHEN {col} IS NULL THEN 1.0
                        ELSE 0.0
                    END
                ),
                2
            ) AS {col}_missing_pct
        """)

    missing_result = con.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            {",".join(expressions)}
        FROM {fact}
    """).df()

    display(missing_result)

else:
    print("No expected metric columns found.")


# ============================================================
# F. MISSINGNESS BY CLIENT
# ============================================================

print("=" * 70)
print("F. MISSINGNESS BY CLIENT")
print("=" * 70)

if "gsc_avg_position" in columns:

    missing_by_client = con.sql(f"""
        SELECT
            client_hash_id,
            COUNT(*) AS rows,
            ROUND(
                100.0 * AVG(
                    CASE
                        WHEN gsc_avg_position IS NULL
                        THEN 1.0
                        ELSE 0.0
                    END
                ),
                2
            ) AS missing_position_pct
        FROM {fact}
        GROUP BY client_hash_id
        ORDER BY missing_position_pct DESC
        LIMIT 20
    """).df()

    display(missing_by_client)
else:
    print("gsc_avg_position not available.")


# ============================================================
# GSC AVG POSITION = 0
# ============================================================

print("=" * 70)
print("G. GSC AVG POSITION = 0")
print("=" * 70)

if "gsc_avg_position" in columns:

    position_check = con.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(*) FILTER (
                WHERE gsc_avg_position = 0
            ) AS zero_position_rows,
            ROUND(
                100.0 *
                COUNT(*) FILTER (
                    WHERE gsc_avg_position = 0
                ) / COUNT(*),
                2
            ) AS zero_position_pct
        FROM {fact}
    """).df()

    display(position_check)
else:
    print("gsc_avg_position not available.")


# ============================================================
# H. FINAL MONTH — JUNE 2026
# ============================================================

print("=" * 70)
print("H. FINAL MONTH — JUNE 2026")
print("=" * 70)

final_month = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS rows
    FROM {fact}
    WHERE report_date >= DATE '2026-06-01'
      AND report_date < DATE '2026-07-01'
""").df()

display(final_month)


# ============================================================
# I. MID-PANEL — MARCH 2026
# ============================================================

print("=" * 70)
print("I. MID-PANEL — MARCH 2026")
print("=" * 70)

march = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) AS rows
    FROM {fact}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

display(march)


# ============================================================
# FINAL STATUS
# ============================================================

print("=" * 70)
print("SECTION 3 VERIFICATION COMPLETE")
print("=" * 70)

print("✓ Counts and date window checked")
print("✓ Grain checked")
print("✓ Client history checked")
print("✓ GSC/GA4 access flags checked")
print("✓ Missingness checked")
print("✓ Missingness by client checked")
print("✓ GSC position = 0 checked")
print("✓ June 2026 final month checked")
print("✓ March 2026 development month checked")

A. COUNTS AND DATE WINDOW


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,content_items,dates,first_date,last_date
0,78835655,70,427292,520,2025-01-27,2026-06-30


B. GRAIN CHECK


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

,first_date,last_date,rows
0,2026-06-01,2026-06-30,11694072


I. MID-PANEL — MARCH 2026


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,rows
0,2026-03-01,2026-03-31,9841378


SECTION 3 VERIFICATION COMPLETE
✓ Counts and date window checked
✓ Grain checked
✓ Client history checked
✓ GSC/GA4 access flags checked
✓ Missingness checked
✓ Missingness by client checked
✓ GSC position = 0 checked
✓ June 2026 final month checked
✓ March 2026 development month checked


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The data supports directional search-performance analysis and decision-support, but it cannot prove that a content refresh caused a change in search performance.

Client history is unbalanced, so different clients have different available history.

Rows before a client's `ga4_data_start` have GA4 fields zero-filled while `ga4_data_available` is FALSE. These zeros should not be interpreted as zero engagement.

Feature and outcome windows must be separated to prevent future information from leaking into the features.

The query-level table contains repeated content-level context fields, so these fields should be read with `ANY_VALUE()` rather than summed.

The final month, June 2026, is treated as a sealed outcome/test period and should not be used while developing label logic.

The data measures observed search performance and cannot by itself establish causation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
The data supports directional search-performance analysis and decision-support, but it cannot prove that a content refresh caused a change in search performance.

Client history is unbalanced, so different clients have different available history.

Rows before a client's `ga4_data_start` have GA4 fields zero-filled while `ga4_data_available` is FALSE. These zeros should not be interpreted as zero engagement.

Feature and outcome windows must be separated to prevent future information from leaking into the features.

The query-level table contains repeated content-level context fields, so these fields should be read with `ANY_VALUE()` rather than summed.

The final month, June 2026, is treated as a sealed outcome/test period and should not be used while developing label logic.

The data measures observed search performance and cannot by itself establish causation.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.